In [0]:

df = spark.table("metadata_governance.bronze.raw_metadata")
df.display()

In [0]:
from pyspark.sql.functions import col, when, expr

tier2_fields = ["column_desc", "table_desc", "data_steward", 
                 "security_classification", "term_subdomain", "certification_level"]

# Count how many Tier 2 fields are non-null per row
completeness_expr = " + ".join([f"CASE WHEN {f} IS NOT NULL THEN 1 ELSE 0 END" for f in tier2_fields])

df_scored = df.withColumn(
    "tier2_filled_count", expr(completeness_expr)
).withColumn(
    "row_completeness_pct", (col("tier2_filled_count") / len(tier2_fields)) * 100
)

df_scored.select("column_id", "table_name", "tier2_filled_count", "row_completeness_pct").display()

In [0]:
from pyspark.sql.functions import avg, round as spark_round

table_completeness = df_scored.groupBy("table_id", "table_name").agg(
    spark_round(avg("row_completeness_pct"), 2).alias("table_completeness_pct")
)

table_completeness.display()


In [0]:
table_maturity = table_completeness.withColumn(
    "maturity_tier",
    when(col("table_completeness_pct") >= 90, "High")
    .when(col("table_completeness_pct") >= 50, "Medium")
    .otherwise("Low")
)

table_maturity.display()

In [0]:
df_pii_check = df.withColumn(
    "pii_non_compliant",
    when((col("pii_flag") == True) & (col("security_classification").isNull()), True)
    .otherwise(False)
)

df_pii_check.filter(col("pii_non_compliant") == True).select(
    "column_id", "column_name", "table_name", "pii_flag", "security_classification"
).display()

In [0]:
df_steward_check = df.withColumn(
    "unowned",
    when(col("data_steward").isNull(), True).otherwise(False)
)

df_steward_check.filter(col("unowned") == True).select(
    "column_id", "column_name", "table_name", "data_steward"
).display()